### 1. Import libraries and load data from database.

In [1]:
# import libraries
import nltk
nltk.download(['punkt', 'wordnet'])

import os
import re
import numpy as np
import pandas as pd
import sqlite3
from sqlalchemy import create_engine,inspect

from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import imblearn
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import hamming_loss, confusion_matrix, classification_report, precision_recall_fscore_support, accuracy_score, precision_score, recall_score, f1_score

import tensorflow as tf

import random
from sklearn.datasets import make_classification
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

c:\Users\sinde\anaconda3\lib\site-packages\scipy\__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sinde\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sinde\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
# Move to datasets folder
original_directory = os.getcwd()
dataset_directory = './dataset'
os.chdir(dataset_directory)

In [3]:
#Look for the tables name in the SQL database
engine = create_engine('sqlite:///DisasterResponse.db')

# Create an inspector
inspector = inspect(engine)

# Get the list of table names
table_names = inspector.get_table_names()

table_names

['messages']

In [4]:
# Import data
engine = create_engine('sqlite:///DisasterResponse.db')
connection = engine.connect()
df = pd.read_sql("SELECT * FROM messages", connection)
connection.close()

In [5]:
# Get target columns
target_columns = [col for col in df.columns if col not in ['message', 'related','id','original','genre']]

#Look only at the confirmed related cases
df_related = df[df['related']==1]

# If there is one parameter with no variance, drop it
columns_to_drop = df_related[target_columns].sum() == 0
columns_to_drop = columns_to_drop[columns_to_drop].index

# New datasets for multi-label classification
X = df_related['message']
Y = df_related[target_columns]
Y = Y.drop(columns=columns_to_drop, axis=1)

### 2. Handling Data Imbalance in Multi-label Classification (MLSMOTE)

In [6]:
def tokenize(text_data):
    """
    Tokenize and lemmatize the given text.
    
    Args:
    text: str, the input text to be tokenized and lemmatized
    
    Returns:
    list: a list of lemmatized, lowercased, and stripped tokens from the input text
    """
    tokens = word_tokenize(text_data)
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(tok).lower().strip() for tok in tokens]


In [19]:
def vectorize_transform(df):
    """
    Vectorize and transform the given dataframe using CountVectorizer and TfidfTransformer.
    
    Args:
    df: pandas.DataFrame, the input dataframe containing text data to be vectorized and transformed
    
    Returns:
    scipy.sparse.csr.csr_matrix: a TF-IDF transformed sparse matrix representation of the input dataframe
    """
    # Create a pipeline for vectorization and transformation
    count_vectorizer  = CountVectorizer(tokenizer=tokenize, ngram_range=(1, 3))
    tfidf_transformer = TfidfTransformer()

    # Vectorize and transform the training data
    text_data_counts  = count_vectorizer.fit_transform(df)
    text_data_tfidf  = tfidf_transformer.fit_transform(text_data_counts)

    return text_data_tfidf, count_vectorizer, tfidf_transformer

In [8]:
def get_tail_label(df):
    """
    Give tail label colums of the given target dataframe
    
    args
    df: pandas.DataFrame, target label df whose tail label has to identified
    
    return
    tail_label: list, a list containing column name of all the tail label
    """
    columns = df.columns
    n = len(columns)
    irpl = np.zeros(n)
    for column in range(n):
        irpl[column] = df[columns[column]].value_counts()[1]
    irpl = max(irpl)/irpl
    mir = np.average(irpl)
    tail_label = []
    for i in range(n):
        if irpl[i] > mir:
            tail_label.append(columns[i])
    return tail_label

In [9]:
def get_index(df):
  """
  give the index of all tail_label rows
  args
  df: pandas.DataFrame, target label df from which index for tail label has to identified
    
  return
  index: list, a list containing index number of all the tail label
  """
  tail_labels = get_tail_label(df)
  index = set()
  for tail_label in tail_labels:
    sub_index = set(df[df[tail_label]==1].index)
    index = index.union(sub_index)
  return list(index)

In [10]:
def get_minority_instace(X, y):
    """
    Give minority dataframe containing all the tail labels
    
    args
    X: pandas.DataFrame, the feature vector dataframe
    y: pandas.DataFrame, the target vector dataframe
    
    return
    X_sub: pandas.DataFrame, the feature vector minority dataframe
    y_sub: pandas.DataFrame, the target vector minority dataframe
    """
    index = get_index(y)
    X_sub = X[X.index.isin(index)].reset_index(drop = True)
    y_sub = y[y.index.isin(index)].reset_index(drop = True)
    return X_sub, y_sub

In [11]:
def nearest_neighbour(X):
    """
    Give index of 5 nearest neighbor of all the instance
    
    args
    X: np.array, array whose nearest neighbor has to find
    
    return
    indices: list of list, index of 5 NN of each element in X
    """
    nbs=NearestNeighbors(n_neighbors=5,metric='euclidean',algorithm='kd_tree').fit(X)
    euclidean,indices= nbs.kneighbors(X)
    return indices

In [12]:
def MLSMOTE(X,y, n_sample, indices2):
    """
    Give the augmented data using MLSMOTE algorithm
    
    args
    X: pandas.DataFrame, input vector DataFrame
    y: pandas.DataFrame, feature vector dataframe
    n_sample: int, number of newly generated sample
    
    return
    new_X: pandas.DataFrame, augmented feature vector data
    target: pandas.DataFrame, augmented target vector data
    """
    # Set variables to for MLSMOTE
    X = pd.DataFrame(X.toarray()) # transform a scipy.sparse._csr.csr_matrix to pandas DataFrame 
    n = len(indices2)
    new_X = np.zeros((n_sample, X.shape[1]))
    target = np.zeros((n_sample, y.shape[1]))

    # Multi-label Classification (MLSMOTE)
    for i in range(n_sample):
        reference = random.randint(0,n-1)
        neighbour = random.choice(indices2[reference,1:])
        all_point = indices2[reference]
        nn_df = y[y.index.isin(all_point)]
        ser = nn_df.sum(axis = 0, skipna = True)
        target[i] = np.array([1 if val>2 else 0 for val in ser])
        ratio = random.random()
        gap = X.loc[reference,:] - X.loc[neighbour,:]
        new_X[i] = np.array(X.loc[reference,:] + ratio * gap)

    # Define new DataFrames for train datasets
    new_X = pd.DataFrame(new_X, columns=X.columns)
    target = pd.DataFrame(target, columns=y.columns)
    target = pd.concat([y, target], axis=0)
    new_X = pd.concat([X, new_X], axis=0)
    new_X = csr_matrix(new_X.values) # transform pandas DataFrame back to scipy.sparse._csr.csr_matrix 
    
    return new_X, target

In [20]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

# Getting minority instance of that datframe
X_sub, y_sub = get_minority_instace(X_train, y_train) # In Multi-label settings, we called labels in the majority as the head labels and labels in minority as tail labels.

# Vectorize the features
X_tfidf, count_vectorizer, tfidf_transformer = vectorize_transform(X_sub)

# Give index of 5 nearest neighbor of all the instances
indice = nearest_neighbour(X_tfidf)

# Applying MLSMOTE to augment the dataframe
X_res, y_res = MLSMOTE(X_tfidf, y_sub, 100, indice)

c:\Users\sinde\anaconda3\lib\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\sinde\anaconda3\lib\site-packages\sklearn\neighbors\_base.py:584: UserWarning: cannot use tree with sparse input: using brute force
  warnings.warn("cannot use tree with sparse input: using brute force")


In [21]:
X_tfidf

<1992x119956 sparse matrix of type '<class 'numpy.float64'>'
	with 211812 stored elements in Compressed Sparse Row format>

In [22]:
X_res

<2092x119956 sparse matrix of type '<class 'numpy.float64'>'
	with 290633 stored elements in Compressed Sparse Row format>

In [23]:
y_res

,request,offer,aid_related,medical_help,medical_products,search_and_rescue,security,military,water,food,...,aid_centers,other_infrastructure,weather_related,floods,storm,fire,earthquake,cold,other_weather,direct_report
0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
96,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
97,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
98,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0


### 3.Testing mutiple models

In [17]:
# Define the pipeline with oversampling
pipeline = Pipeline([
    ('clf', MultiOutputClassifier(estimator=RandomForestClassifier()))
])

# Define the parameter grid
param_grid = [
    {
        'clf__estimator__n_estimators': [100, 200],
        'clf__estimator__min_samples_split': [2, 5]
    },
    {
        'clf__estimator': [LogisticRegression(max_iter=1000)],
        'clf__estimator__C': [0.1, 1, 10],
        'clf__estimator__solver': ['liblinear', 'saga']
    }
]

In [36]:
grid_search = GridSearchCV(pipeline, param_grid, cv=2, scoring='precision_weighted', n_jobs=1, verbose=2)
grid_search.fit(X_res, y_res)

# Print the best parameters and best score
print(f'Best parameters found: {grid_search.best_params_}')
print(f'Best Precision score: {grid_search.best_score_}')

Fitting 2 folds for each of 10 candidates, totalling 20 fits


c:\Users\sinde\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[CV] END clf__estimator__min_samples_split=2, clf__estimator__n_estimators=100; total time= 1.8min
[CV] END clf__estimator__min_samples_split=2, clf__estimator__n_estimators=100; total time= 2.2min


c:\Users\sinde\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[CV] END clf__estimator__min_samples_split=2, clf__estimator__n_estimators=200; total time= 4.4min
[CV] END clf__estimator__min_samples_split=2, clf__estimator__n_estimators=200; total time= 4.7min


...

[CV] END clf__estimator=LogisticRegression(max_iter=1000), clf__estimator__C=1, clf__estimator__solver=saga; total time=  16.5s


c:\Users\sinde\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[CV] END clf__estimator=LogisticRegression(max_iter=1000), clf__estimator__C=10, clf__estimator__solver=liblinear; total time=   1.5s
[CV] END clf__estimator=LogisticRegression(max_iter=1000), clf__estimator__C=10, clf__estimator__solver=liblinear; total time=   0.9s


c:\Users\sinde\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[CV] END clf__estimator=LogisticRegression(max_iter=1000), clf__estimator__C=10, clf__estimator__solver=saga; total time=  51.0s
[CV] END clf__estimator=LogisticRegression(max_iter=1000), clf__estimator__C=10, clf__estimator__solver=saga; total time=  23.5s
Best parameters found: {'clf__estimator__min_samples_split': 2, 'clf__estimator__n_estimators': 100}
Best Precision score: 0.7134265650049929


In [24]:
# Define the vectorize function for test data
def vectorize_test(text_data, count_vectorizer, tfidf_transformer):
    text_data_counts = count_vectorizer.transform(text_data)
    text_data_tfidf = tfidf_transformer.transform(text_data_counts)

    return text_data_tfidf

In [25]:
#Considering performance (time and criteria 'gini index' for RandomForestClassifier Machine Learning Model)
pipeline.set_params(clf__estimator__min_samples_split = 2, clf__estimator__n_estimators = 100)

# Train the model
pipeline.fit(X_res, y_res)

# Transform the test set
X_test_tfidf = vectorize_test(X_test, count_vectorizer, tfidf_transformer)

# Predict on test data
Y_pred = pipeline.predict(X_test_tfidf)

# Initialize lists to store the precision, recall, and f1-score for each label
precision_list = []
recall_list = []
f1_list = []

# Calculate precision, recall, and f1-score for each label
for i, column in enumerate(Y.columns):
    precision = precision_score(y_test[column], Y_pred[:, i], average='weighted', zero_division=0)
    recall = recall_score(y_test[column], Y_pred[:, i], average='weighted', zero_division=0)
    f1 = f1_score(y_test[column], Y_pred[:, i], average='weighted', zero_division=0)
    
    precision_list.append(precision)
    recall_list.append(recall)
    f1_list.append(f1)

# Compute macro averages
precision_macro = np.mean(precision_list)
recall_macro = np.mean(recall_list)
f1_macro = np.mean(f1_list)

# Overall metrics
overall_accuracy = (Y_pred == y_test).mean().mean()

print(f'Overall Accuracy: {overall_accuracy:.4f}')
print(f'Macro Average Precision: {precision_macro:.4f}')
print(f'Macro Average Recall: {recall_macro:.4f}')
print(f'Macro Average F1 Score: {f1_macro:.4f}')

Overall Accuracy: 0.9146
Macro Average Precision: 0.9063
Macro Average Recall: 0.9146
Macro Average F1 Score: 0.8928


# One Function

In [7]:
# import libraries
import os
import re
import random
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

import nltk
nltk.download(['punkt', 'wordnet'])
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import random
from sklearn.datasets import make_classification
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

# necessary functions for train dataset
def tokenize(text_data):
    """
    Tokenize and lemmatize the given text.
    
    Args:
    text: str, the input text to be tokenized and lemmatized
    
    Returns:
    list: a list of lemmatized, lowercased, and stripped tokens from the input text
    """
    tokens = word_tokenize(text_data)
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(tok).lower().strip() for tok in tokens]

def vectorize_transform(df):
    """
    Vectorize and transform the given dataframe using CountVectorizer and TfidfTransformer.
    
    Args:
    df: pandas.DataFrame, the input dataframe containing text data to be vectorized and transformed
    
    Returns:
    scipy.sparse.csr.csr_matrix: a TF-IDF transformed sparse matrix representation of the input dataframe
    """
    # Create a pipeline for vectorization and transformation
    count_vectorizer  = CountVectorizer(tokenizer=tokenize, token_pattern=None, ngram_range=(1, 3))
    tfidf_transformer = TfidfTransformer()

    # Vectorize and transform the training data
    text_data_counts  = count_vectorizer.fit_transform(df)
    text_data_tfidf  = tfidf_transformer.fit_transform(text_data_counts)

    return text_data_tfidf, count_vectorizer, tfidf_transformer

def get_tail_label(df):
    """
    Give tail label colums of the given target dataframe
    
    args
    df: pandas.DataFrame, target label df whose tail label has to identified
    
    return
    tail_label: list, a list containing column name of all the tail label
    """
    # In Multi-label settings, 
    # we called labels in the majority as the head labels and labels in minority as tail labels.
    columns = df.columns
    n = len(columns)
    irpl = np.zeros(n)
    for column in range(n):
        irpl[column] = df[columns[column]].value_counts()[1]
    irpl = max(irpl)/irpl
    mir = np.average(irpl)
    tail_label = []
    for i in range(n):
        if irpl[i] > mir:
            tail_label.append(columns[i])
    return tail_label

def get_index(df):
  """
  give the index of all tail_label rows
  args
  df: pandas.DataFrame, target label df from which index for tail label has to identified
    
  return
  index: list, a list containing index number of all the tail label
  """
  tail_labels = get_tail_label(df)
  index = set()
  for tail_label in tail_labels:
    sub_index = set(df[df[tail_label]==1].index)
    index = index.union(sub_index)
  return list(index)

def get_minority_instace(X, y):
    """
    Give minority dataframe containing all the tail labels
    
    args
    X: pandas.DataFrame, the feature vector dataframe
    y: pandas.DataFrame, the target vector dataframe
    
    return
    X_sub: pandas.DataFrame, the feature vector minority dataframe
    y_sub: pandas.DataFrame, the target vector minority dataframe
    """
    index = get_index(y)
    X_sub = X[X.index.isin(index)].reset_index(drop = True)
    y_sub = y[y.index.isin(index)].reset_index(drop = True)
    return X_sub, y_sub

def nearest_neighbour(X):
    """
    Give index of 5 nearest neighbor of all the instance
    
    args
    X: np.array, array whose nearest neighbor has to find
    
    return
    indices: list of list, index of 5 NN of each element in X
    """
    nbs=NearestNeighbors(n_neighbors=5,metric='euclidean',algorithm='kd_tree').fit(X)
    euclidean,indices= nbs.kneighbors(X)
    return indices

def MLSMOTE(X,y, n_sample, indices2):
    """
    Give the augmented data using MLSMOTE algorithm
    
    args
    X: pandas.DataFrame, input vector DataFrame
    y: pandas.DataFrame, feature vector dataframe
    n_sample: int, number of newly generated sample
    
    return
    new_X: pandas.DataFrame, augmented feature vector data
    target: pandas.DataFrame, augmented target vector data
    """
    # Set variables to for MLSMOTE
    X = pd.DataFrame(X.toarray()) # transform a scipy.sparse._csr.csr_matrix to pandas DataFrame 
    n = len(indices2)
    new_X = np.zeros((n_sample, X.shape[1]))
    target = np.zeros((n_sample, y.shape[1]))

    # Multi-label Classification (MLSMOTE)
    for i in range(n_sample):
        reference = random.randint(0,n-1)
        neighbour = random.choice(indices2[reference,1:])
        all_point = indices2[reference]
        nn_df = y[y.index.isin(all_point)]
        ser = nn_df.sum(axis = 0, skipna = True)
        target[i] = np.array([1 if val>2 else 0 for val in ser])
        ratio = random.random()
        gap = X.loc[reference,:] - X.loc[neighbour,:]
        new_X[i] = np.array(X.loc[reference,:] + ratio * gap)

    # Define new DataFrames for train datasets
    new_X = pd.DataFrame(new_X, columns=X.columns)
    target = pd.DataFrame(target, columns=y.columns)
    target = pd.concat([y, target], axis=0)
    new_X = pd.concat([X, new_X], axis=0)
    new_X = csr_matrix(new_X.values) # transform pandas DataFrame back to scipy.sparse._csr.csr_matrix 
    
    return new_X, target

# necessary functions for test dataset
def vectorize_test(text_data, count_vectorizer, tfidf_transformer):
    text_data_counts = count_vectorizer.transform(text_data)
    text_data_tfidf = tfidf_transformer.transform(text_data_counts)

    return text_data_tfidf

###### Main function

# Move to datasets folder
original_directory = os.getcwd()
dataset_directory = './dataset'
os.chdir(dataset_directory)

# Import data
engine = create_engine('sqlite:///DisasterResponse.db')
with engine.connect() as connection:
    df = pd.read_sql("SELECT * FROM messages", connection)
engine.dispose()

#return to original directory
os.chdir(original_directory)

# Get target columns
target_columns = [col for col in df.columns if col not in ['message', 'related','id','original','genre']]

#Look only at the confirmed related cases
df_related = df[df['related']==1]

# If there is one parameter with no variance, drop it
columns_to_drop = df_related[target_columns].sum() == 0
columns_to_drop = columns_to_drop[columns_to_drop].index

# New datasets for multi-label classification
X = df_related['message']
Y = df_related[target_columns]
Y = Y.drop(columns=columns_to_drop, axis=1)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

# Getting minority instance (tail labels) of that datframe
X_sub, y_sub = get_minority_instace(X_train, y_train) 

# Vectorize the features
X_tfidf, count_vectorizer, tfidf_transformer = vectorize_transform(X_sub)

# Give index of 5 nearest neighbor of all the instances
indice = nearest_neighbour(X_tfidf)

# Applying MLSMOTE to augment the dataframe
X_res, y_res = MLSMOTE(X_tfidf, y_sub, 100, indice)

# Transform the test set
X_test_tfidf = vectorize_test(X_test, count_vectorizer, tfidf_transformer)

# Define the pipeline with oversampling
pipeline = Pipeline([
    ('clf', MultiOutputClassifier(estimator=RandomForestClassifier()))
])

#Considering performance (time and criteria 'gini index' for RandomForestClassifier Machine Learning Model)
pipeline.set_params(clf__estimator__min_samples_split = 2, clf__estimator__n_estimators = 100)

# Train the model
pipeline.fit(X_res, y_res)

# Predict on test data
Y_pred = pipeline.predict(X_test_tfidf)

# Initialize lists to store the precision, recall, and f1-score for each label
precision_list = []
recall_list = []
f1_list = []

# Calculate precision, recall, and f1-score for each label
for i, column in enumerate(Y.columns):
    precision = precision_score(y_test[column], Y_pred[:, i], average='weighted', zero_division=0)
    recall = recall_score(y_test[column], Y_pred[:, i], average='weighted', zero_division=0)
    f1 = f1_score(y_test[column], Y_pred[:, i], average='weighted', zero_division=0)
    
    precision_list.append(precision)
    recall_list.append(recall)
    f1_list.append(f1)

# Compute macro averages
precision_macro = np.mean(precision_list)
recall_macro = np.mean(recall_list)
f1_macro = np.mean(f1_list)

# Overall metrics
overall_accuracy = (Y_pred == y_test).mean().mean()

print(f'Overall Accuracy: {overall_accuracy:.4f}')
print(f'Macro Average Precision: {precision_macro:.4f}')
print(f'Macro Average Recall: {recall_macro:.4f}')
print(f'Macro Average F1 Score: {f1_macro:.4f}')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sinde\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sinde\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


AttributeError: 'NoneType' object has no attribute 'split'